# Noisy devices and error mitigation

QSO 1.4 simulates noise the way devices have it and mitigates it the way
experiments do:

- device models with relaxation, gate errors and readout errors;
- quantum trajectories for registers beyond density matrices;
- estimates with error bars;
- readout mitigation, zero-noise extrapolation, probabilistic error
  cancellation and Pauli twirling;
- classical shadows.

Every number below comes with its standard error, and every mitigated
value is compared with the noiseless one.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import metatron_qso as q

print("QSO", q.__version__)

## A device

A five-qubit toy device has the following noise:

- T₁ and T₂ on every qubit, and qubit 3 is worse;
- readout errors;
- a depolarising error and a small coherent ZZ rotation on every CX;
- small over-rotations on one-qubit gates.

Durations and relaxation times share one unit of time.

In [ ]:
device = (
    q.DeviceModel("toy", num_qubits=5)
    .with_default_qubit(t1=100.0, t2=80.0, readout=(0.02, 0.045))
    .with_qubit(3, t1=60.0, t2=45.0, readout=(0.03, 0.07))
    .with_rule(gate="cx", duration=0.3, depolarizing=0.008,
               channels=[q.Channel.rotation("ZZ", 0.02)])
    .with_rule(num_qubits=1, duration=0.035, depolarizing=0.0005, over_rotation=0.002)
)
print(device)
print(device.to_json()[:300], "...")

The test circuit prepares a five-qubit GHZ state. The observable is
H = Σ Z_i Z_(i+1), whose noiseless value is 4.

In [ ]:
n = 5
ghz = q.Circuit(n).h(0)
for k in range(n - 1):
    ghz.cx(k, k + 1)
H = q.PauliSum(" + ".join(f"Z{k} Z{k + 1}" for k in range(n - 1)))
ideal = H.expectation(q.simulate(ghz))
rho = q.DensityMatrix.simulate(ghz, device=device)
print("noiseless", ideal, " noisy (exact)", rho.expectation(H), " purity", rho.purity())

## Trajectories converge to the density matrix

A trajectory evolves one state vector and applies one Kraus operator of
every channel, with its Born probability. Averages converge to the
density-matrix value with standard error ∝ 1/√N.

In [ ]:
exact = rho.expectation(H)
Ns = [50, 200, 800, 3200]
ests = [q.TrajectorySimulator(device, N, seed=1).expectation(ghz, H) for N in Ns]
plt.errorbar(Ns, [e.value for e in ests], yerr=[e.std_error for e in ests], fmt="o", capsize=3)
plt.axhline(exact, color="k", lw=1, label="density matrix")
plt.xscale("log"); plt.xlabel("trajectories"); plt.ylabel("⟨H⟩"); plt.legend()
plt.show()
for N, e in zip(Ns, ests):
    print(f"{N:5d}  {e.value:.4f} ± {e.std_error:.4f}")

## Estimates from shots

Real measurements add shot noise and readout errors. `estimate` returns an
`Estimate` with the standard error from the per-shot values. With several
measurement groups, the shot allocation matters: Neyman allocation
(`"adaptive"`) puts the shots where the spread is.

In [ ]:
c2 = q.Circuit(2).h(0).h(1)
h2 = q.PauliSum("5 Z0 Z1 + 0.1 X0 X1 + 0.1 Y0 Y1 + 0.1 X0 Y1")
for allocation in ["uniform", "weighted", "adaptive"]:
    e, groups = q.estimate(c2, h2, 4000, allocation=allocation, seed=2)
    print(f"{allocation:9s} {e.value:+.4f} ± {e.std_error:.4f}   shots per group {[g['shots'] for g in groups]}")

## Readout mitigation

A tensored calibration measures the per-qubit readout errors. The
calibration then corrects every measured parity.

In [ ]:
cal = q.ReadoutCalibration.tensored(n, 100_000, seed=3, device=device)
print("measured readout errors:", [tuple(round(x, 4) for x in e) for e in cal.qubit_errors()])
raw, _ = q.estimate(ghz, H, 200_000, device=device, seed=4)
mit, _ = q.estimate(ghz, H, 200_000, device=device, seed=4, readout_calibration=cal)
print(f"raw        {raw.value:.4f} ± {raw.std_error:.4f}")
print(f"mitigated  {mit.value:.4f} ± {mit.std_error:.4f}   (exact without readout error {exact:.4f})")

## Zero-noise extrapolation

Folding (`G → G G† G`) amplifies the gate noise without changing the
ideal circuit. The extrapolation to zero noise then removes most of the
remaining error. Readout errors do not scale with folding, so ZNE runs on
top of readout mitigation here.

In [ ]:
seed = iter(range(100, 200))
def run(folded):
    return q.estimate(folded, H, 200_000, device=device, seed=next(seed), readout_calibration=cal)[0]

r = q.zne(ghz, [1.0, 3.0, 5.0], run)
scales = r["scales"]; values = r["values"]
plt.errorbar(scales, [v.value for v in values], yerr=[v.std_error for v in values], fmt="o", capsize=3, label="folded")
plt.errorbar([0], [r["estimate"].value], yerr=[r["estimate"].std_error], fmt="s", capsize=3, label="extrapolated")
xs = np.linspace(0, 5, 50)
plt.plot(xs, np.polyval(np.polyfit(scales, [v.value for v in values], 2), xs), "--", lw=1)
plt.axhline(ideal, color="k", lw=1, label="noiseless")
plt.xlabel("noise scale"); plt.ylabel("⟨H⟩"); plt.legend(); plt.show()
print(f"ZNE: {r['estimate'].value:.4f} ± {r['estimate'].std_error:.4f}  (noiseless {ideal})")

## Probabilistic error cancellation

PEC inverts Pauli noise exactly, at a sampling cost Γ. The device's CX
noise as Pauli channels:

In [ ]:
pauli_device = (
    q.DeviceModel("toy, Pauli noise")
    .with_rule(gate="cx", depolarizing=0.008, channels=[q.Channel.pauli({"ZZ": 0.002})])
    .with_rule(num_qubits=1, depolarizing=0.0005)
)
noisy = q.DensityMatrix.simulate(ghz, device=pauli_device).expectation(H)
r = q.pec_expectation(ghz, H, pauli_device, samples=20_000, seed=5)
print(f"noisy {noisy:.4f}  PEC {r['estimate'].value:.4f} ± {r['estimate'].std_error:.4f}  overhead Γ = {r['overhead']:.3f}")

## Pauli twirling

The device's coherent ZZ error accumulates as an angle. Twirling the CX
gates turns it into stochastic Pauli noise. Averaged over twirls, the
result matches the twirled channel.

In [ ]:
coherent = q.DeviceModel("zz").with_rule(gate="cx", channels=[q.Channel.rotation("ZZ", 0.3)])
twirled = q.DeviceModel("zz, twirled").with_rule(gate="cx", channels=[q.Channel.rotation("ZZ", 0.3).twirled()])
c = q.Circuit(2).h(0).cx(0, 1).rz(1, 0.3).cx(0, 1).h(0)
z = q.PauliSum("Z0")
avg = q.twirled_average(c, 400, lambda t: q.DensityMatrix.simulate(t, device=coherent).expectation(z), seed=6)
print("coherent", q.DensityMatrix.simulate(c, device=coherent).expectation(z))
print("twirled average", avg.value, "±", avg.std_error)
print("twirled channel", q.DensityMatrix.simulate(c, device=twirled).expectation(z))

## Classical shadows

One set of random-basis snapshots estimates many observables at once. Here
the snapshots come from the ideal GHZ state. Shadows of a device can be
collected the same way with `ClassicalShadow.collect(..., device=...)`, and
then include the device's noise and readout errors. A Pauli string on k
qubits has single-snapshot variance 3ᵏ − ⟨P⟩², so the five-body string is
the least precise.

In [ ]:
psi = q.simulate(ghz)
shadow = q.ClassicalShadow.from_state(psi, 30_000, seed=7)
for label in ["Z0 Z1", "Z1 Z2", "Z2 Z3", "Z3 Z4", "X0 X1 X2 X3 X4"]:
    h = q.PauliSum(label)
    e = shadow.estimate(h)
    print(f"{label:16s} {e.value:+.3f} ± {e.std_error:.3f}   exact {h.expectation(psi):+.3f}")

## Beyond density matrices

Trajectories reach any register a state vector holds. Here a 16-qubit GHZ
state on a uniform device runs in seconds.

In [ ]:
uniform = (
    q.DeviceModel("uniform")
    .with_default_qubit(t1=100.0, t2=80.0)
    .with_rule(gate="cx", duration=0.3, depolarizing=0.008)
    .with_rule(num_qubits=1, duration=0.035, depolarizing=0.0005)
)
big = 16
ghz16 = q.Circuit(big).h(0)
for k in range(big - 1):
    ghz16.cx(k, k + 1)
H16 = q.PauliSum(" + ".join(f"Z{k} Z{k + 1}" for k in range(big - 1)))
e = q.TrajectorySimulator(uniform, 200, seed=8).expectation(ghz16, H16)
print(f"⟨H⟩ = {e.value:.3f} ± {e.std_error:.3f}  (noiseless {big - 1})")